<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Temperature_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map.add("basemap_selector")
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
border = (
    ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
    .filterBounds(roi)
    .geometry().simplify(100)
)
vis_params = {"color": "yellow"}
map.addLayer(border, vis_params, "border")

In [ ]:
gpm = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate('2026-01-01','2026-04-01')
    .select('tasmin_mean')
    .map(
        lambda img: img.clip(border).copyProperties(img, ['system:time_start'])
    )
)
gpm.getInfo()

In [ ]:
import xarray as xr

In [ ]:
temperature_ds = xr.open_dataset(
    gpm,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 100,
    geometry = border
)

temperature_ds

In [ ]:
temperature_ds = temperature_ds.sortby("time")
temperature_ds

In [ ]:
import numpy as np

In [ ]:
temperature_monthly = temperature_ds.resample(time = 'M').mean('time')
temperature_monthly = xr.where(temperature_monthly == 0,  np.nan, temperature_monthly)

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
g = temperature_monthly.tasmin_mean.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 3,
    robust = True,
    level = 10,
    cmap = "turbo_r"
)
for ax in g.axes.flat:
    ax.set_aspect('equal')

monthly_temp_change = temperature_monthly.diff('time')

In [ ]:
print(f"Minimum temperature (tasmin_mean): {temperature_monthly.tasmin_mean.min().values}")
print(f"Maximum temperature (tasmin_mean): {temperature_monthly.tasmin_mean.max().values}")
print(f"Number of non-null values: {temperature_monthly.tasmin_mean.count().values}")